# – Part 2: Data Preprocessing


## Step 0 – Install Required Libraries

In [1]:
! gdown --id 1v00MuC_DxJx2Jsh785qEcu1mi7GTdD31

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1v00MuC_DxJx2Jsh785qEcu1mi7GTdD31
To: /content/final_dataset_text_label.csv
100% 2.42M/2.42M [00:00<00:00, 21.3MB/s]


In [2]:
!pip install camel-tools pyarabic nltk

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
import nltk
from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar
from camel_tools.tokenizers.word import simple_word_tokenize
from pyarabic.araby import strip_tashkeel
import pyarabic.araby as araby
nltk.download('stopwords')
print('All libraries installed successfully!')

All libraries installed successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## Step 1a – Load and Explore the Dataset

In [4]:
FILE_PATH = 'final_dataset_text_label.csv'

df = pd.read_csv(FILE_PATH)
print('Shape:', df.shape)
df.head()

Shape: (1495, 2)


,text,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,human
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,human
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,human
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,human
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,human


In [5]:
# Basic exploration
print('=== Column names ===')
print(df.columns.tolist())

print('\n=== Label distribution ===')
print(df['label'].value_counts())

print('\n=== Missing values ===')
print(df.isnull().sum())


=== Column names ===
['text', 'label']

=== Label distribution ===
label
ai                     678
human                  660
ai generated            40
ai-generated            20
humen                   20
collected               15
human-written           10
ai_generation           10
ai_generated            10
hand written            10
human_written            5
human(collected)         5
human(hand_written)      5
collect                  5
label                    1
al                       1
Name: count, dtype: int64

=== Missing values ===
text     0
label    0
dtype: int64


---
## Step 1b – Label Normalization


In [6]:
#  Label Standardization

LABEL_MAP = {
    'ai'              : 'ai_generated',
    'ai generated'    : 'ai_generated',
    'ai-generated'    : 'ai_generated',
    'ai_generation'   : 'ai_generated',
    'ai_generated'    : 'ai_generated',
    'al'              : 'ai_generated',

    'human'               : 'human_written',
    'humen'               : 'human_written',
    'human-written'       : 'human_written',
    'human_written'       : 'human_written',
    'hand written'        : 'human_written',
    'human written'       : 'human_written',
    'human(hand_written)' : 'human_written',
    'human(collected)'    : 'human_written',
    'collect'             : 'human_written',
    'collected'           : 'human_written',
}

df['label'] = df['label'].astype(str).str.strip().str.lower()

before = len(df)
df = df[df['label'] != 'label'].reset_index(drop=True)
print(f'Removed {before - len(df)} row(s) with label="label"')

df['label'] = df['label'].map(LABEL_MAP)

unmapped = df['label'].isna().sum()
if unmapped > 0:
    print(f'Warning: {unmapped} row(s) had unrecognised labels and were dropped.')
    df = df.dropna(subset=['label']).reset_index(drop=True)

print('\n=== Clean Label Distribution ===')
print(df['label'].value_counts())
print(f'\nTotal samples after label cleaning: {len(df)}')


Removed 1 row(s) with label="label"

=== Clean Label Distribution ===
label
ai_generated     759
human_written    735
Name: count, dtype: int64

Total samples after label cleaning: 1494


---
## Step 2 – Text Cleaning



In [7]:
ARABIC_STOPWORDS = set(stopwords.words('arabic'))

EXTRA_STOPWORDS = {'هذا', 'هذه', 'ذلك', 'تلك', 'هناك', 'حيث', 'كما', 'أيضا',
                   'بعض', 'كل', 'جميع', 'أي', 'لكن', 'إلا', 'لأن', 'بسبب'}
ARABIC_STOPWORDS.update(EXTRA_STOPWORDS)

def clean_arabic_text(text: str) -> str:
    if not isinstance(text, str):
        return ''

    # Remove diacritics (tashkeel)
    text = araby.strip_tashkeel(text)

    # Remove tatweel (kashida elongation)
    text = araby.strip_tatweel(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove dates (DD/MM/YYYY, YYYY-MM-DD, etc.)
    text = re.sub(r'\b(\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4})\b', ' ', text)

    # Remove Arabic-Indic digit dates
    text = re.sub(
        r'[\u0660-\u0669]{1,4}[-/.]([\u0660-\u0669]{1,2})[-/.]([\u0660-\u0669]{1,4})',
        ' ', text)

    # Remove punctuation and ALL non-Arabic characters
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # Remove remaining standalone digits (Arabic-Indic / Western)
    text = re.sub(r'[\u0660-\u0669\u06F0-\u06F90-9]+', ' ', text)

    # Remove Arabic punctuation marks (،  ؛  ؟  ۔  ٪)
    text = re.sub(r'[\u060C\u061B\u061F\u06D4\u066A\u066B\u066C]', ' ', text)

    # Remove extra whitespace / newlines
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stop words (optional, we will try both ways)
    #tokens = text.split()
    #tokens = [tok for tok in tokens if tok not in ARABIC_STOPWORDS]
    #text = ' '.join(tokens)

    return text


sample_raw = df['text'].iloc[9]
sample_clean = clean_arabic_text(sample_raw)
print('\n[Before]', sample_raw[:200])
print('\n[After] ', sample_clean[:200])



[Before] ظهر الأمن السيراني مع نهاية الحرب الباردة، وظهور مصطلح حرب الإنترنت أو الحرب السيبرانية، التي جاءت مع بداية اعتماد الدول على أجهزة الكمبيوتر في مؤسساتها وتطوير وحدة المعالجة المركزية في هذه الأجهزة، ا

[After]  ظهر الأمن السيراني نهاية الحرب الباردة وظهور مصطلح حرب الإنترنت الحرب السيبرانية جاءت بداية اعتماد الدول أجهزة الكمبيوتر مؤسساتها وتطوير وحدة المعالجة المركزية الأجهزة دخلت عمل المؤسسات والحكومات وحتى


In [8]:
# Apply cleaning to the entire dataset (after we check on sample)
df['text_clean'] = df['text'].apply(clean_arabic_text)

before = len(df)
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)
after = len(df)
print(f'Rows before: {before} | Rows after: {after} | Dropped: {before - after}')
df[['text', 'text_clean', 'label']].head(3)

Rows before: 1494 | Rows after: 1494 | Dropped: 0


,text,text_clean,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,زيادة الاعتماد الخدمات الرقمية أصبحت حماية الب...,human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات تحدث وتتم بسرعة وسهولة قلة الوعي ...,human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,ظل التطور التكنولوجي الكبير العصر وزيادة الهجم...,human_written


---
## Step 3 – Normalization (Word Variant Handling)


In [ ]:
def normalize_arabic(text: str) -> str:
    """Normalize Arabic word variants. Returns a clean string."""
    if not isinstance(text, str):
        return ''

    # Normalize Alef variants → plain Alef
    text = re.sub(r'[أإآٱ]', 'ا', text)

    

    return text

df['text_normalized'] = df['text_clean'].apply(normalize_arabic)

print('text_normalized dtype:', df['text_normalized'].dtype)
print('Sample (string):', df['text_normalized'].iloc[0][:150])


text_normalized dtype: object
Sample (string): زياده الاعتماد الخدمات الرقميه اصبحت حمايه البيانات الشخصيه لكل منا اهم التحديات تواجهنا تشهد الهجمات الالكترونيه تطورا سريعا السنوات الاخيره يستخدم ا


---
## Step 4 – Tokenization


In [10]:
df['tokens_normalized'] = df['text_normalized'].apply(
    lambda t: simple_word_tokenize(t) if isinstance(t, str) else []
)
df['token_count'] = df['tokens_normalized'].apply(len)

print('\nSample tokens (first row):')
print(df['tokens_normalized'].iloc[0][:20])



Sample tokens (first row):
['زياده', 'الاعتماد', 'الخدمات', 'الرقميه', 'اصبحت', 'حمايه', 'البيانات', 'الشخصيه', 'لكل', 'منا', 'اهم', 'التحديات', 'تواجهنا', 'تشهد', 'الهجمات', 'الالكترونيه', 'تطورا', 'سريعا', 'السنوات', 'الاخيره']


---
## Step 5 – Final Dataset Summary & Save

In [11]:
print('=== Final Dataset Overview ===')
print('\nDataFrame columns:')
print(df.columns.tolist())
df[['text', 'text_clean', 'text_normalized', 'tokens_normalized', 'label']].head(5)

=== Final Dataset Overview ===

DataFrame columns:
['text', 'label', 'text_clean', 'text_normalized', 'tokens_normalized', 'token_count']


,text,text_clean,text_normalized,tokens_normalized,label
0,مع زيادة الاعتماد على الخدمات الرقمية أصبحت حم...,زيادة الاعتماد الخدمات الرقمية أصبحت حماية الب...,زياده الاعتماد الخدمات الرقميه اصبحت حمايه الب...,"[زياده, الاعتماد, الخدمات, الرقميه, اصبحت, حما...",human_written
1,معظم الهجمات التي تحدث وتتم بسرعة وسهولة هي بس...,معظم الهجمات تحدث وتتم بسرعة وسهولة قلة الوعي ...,معظم الهجمات تحدث وتتم بسرعه وسهوله قله الوعي ...,"[معظم, الهجمات, تحدث, وتتم, بسرعه, وسهوله, قله...",human_written
2,في ظل التطور التكنولوجي الكبير في هذا العصر وز...,ظل التطور التكنولوجي الكبير العصر وزيادة الهجم...,ظل التطور التكنولوجي الكبير العصر وزياده الهجم...,"[ظل, التطور, التكنولوجي, الكبير, العصر, وزياده...",human_written
3,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الأمن السيبراني بالذكاء الاصطناعي ارتباط...,يرتبط الامن السيبراني بالذكاء الاصطناعي ارتباط...,"[يرتبط, الامن, السيبراني, بالذكاء, الاصطناعي, ...",human_written
4,الأمن السيبراني هو أحد أهم المجالات في العصر ا...,الأمن السيبراني أهم المجالات العصر الحديث لأنه...,الامن السيبراني اهم المجالات العصر الحديث لانه...,"[الامن, السيبراني, اهم, المجالات, العصر, الحدي...",human_written


In [12]:
TEXT_OUTPUT_FILE = 'text_preprocessed_dataset.csv'
df[['text_normalized', 'label']].to_csv(TEXT_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TEXT_OUTPUT_FILE}')

TOKENS_OUTPUT_FILE = 'tokens_preprocessed_dataset.csv'
df[['tokens_normalized', 'label']].to_csv(TOKENS_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved to {TOKENS_OUTPUT_FILE}')


Saved to text_preprocessed_dataset.csv
Saved to tokens_preprocessed_dataset.csv
